# Dataflow Gen2 Analyzer (run inside Fabric)

Checks Dataflow Gen2 items against Microsoft's [performance best practices](https://learn.microsoft.com/en-us/fabric/data-factory/dataflow-gen2-performance-best-practices) and the [copy activity / dataflow / Spark decision guide](https://learn.microsoft.com/en-us/fabric/fundamentals/decision-guide-pipeline-dataflow-spark), then recommends fixes and alternative engines.

**Before you run**
1. Attach a **default Lakehouse** (reports are saved to its `Files/dfg2_reports/`). Optional but recommended.
2. You need at least **Contributor** on every workspace you scan (read-only analysis, nothing is changed).
3. Works in a Spark or a Python notebook. Authentication uses **your own identity** (no secrets).

Run the cells top to bottom. Edit only the **Parameters** cell.

In [ ]:
# 1) Install the analyzer into this session (no git needed).
%pip install --quiet "https://github.com/SoomroFarhanH/SemanticModelBPforAI/archive/refs/heads/main.zip#subdirectory=Analyzer"

# Offline / restricted-network alternative: upload the Analyzer folder to the lakehouse (Files/Analyzer) and use
# import sys; sys.path.insert(0, "/lakehouse/default/Files/Analyzer")

In [ ]:
# 2) Parameters ------------------------------------------------------------------
WORKSPACES      = []        # workspace names or ids, e.g. ["Sales Analytics", "Finance"]. Empty + ALL_WORKSPACES=False scans the current workspace only.
ALL_WORKSPACES  = False     # True = every workspace you can access (can take a while)
DATAFLOW_FILTER = None      # only dataflows whose name contains this text
HISTORY_DAYS    = 30        # refresh history window
INCLUDE_LEGACY  = False     # also read non-CI/CD Gen2 / Gen1 dataflows through the Power BI API
CONFIG_FILE     = None      # optional JSON file with thresholds, e.g. "/lakehouse/default/Files/dfg2_config.json"
SAVE_TO_TABLES  = False     # True = also append scorecard + findings to Delta tables in the default lakehouse (Spark notebook)
EXPORTED_PATHS  = []        # optional: folders/files of exported definitions, e.g. ["/lakehouse/default/Files/exported_dataflows"]

if not WORKSPACES and not ALL_WORKSPACES and not EXPORTED_PATHS:
    import sempy.fabric as _fabric   # preinstalled in Fabric runtimes
    WORKSPACES = [_fabric.get_workspace_id()]
    print("Scanning the current workspace:", WORKSPACES[0])

In [ ]:
# 3) Run the analysis
from dfg2_advisor.notebook import run, scorecard_rows, findings_rows

out = run(
    workspaces=WORKSPACES, all_workspaces=ALL_WORKSPACES, dataflow=DATAFLOW_FILTER, paths=EXPORTED_PATHS,
    history_days=HISTORY_DAYS, include_legacy=INCLUDE_LEGACY, config=CONFIG_FILE,
)
results = out["results"]
print("\nReports saved to:", out["out_dir"])
for fmt, p in out["paths"].items():
    print(f"  {fmt}: {p}")

In [ ]:
# 4) Scorecard: lowest score first. 'None' score = definition could not be read (see DFG-000 in the report).
import pandas as pd
score_df = pd.DataFrame(scorecard_rows(results)).sort_values("score", na_position="last")
try:
    display(score_df)
except NameError:
    print(score_df.to_string())

In [ ]:
# 5) Findings with remediation steps (filter as needed)
findings_df = pd.DataFrame(findings_rows(results))
order = {"critical": 0, "high": 1, "medium": 2, "low": 3, "info": 4}
if not findings_df.empty:
    findings_df = findings_df.sort_values(by="severity", key=lambda s: s.map(order))
try:
    display(findings_df)
except NameError:
    print(findings_df.head(50).to_string())

In [ ]:
# 6) Open the full interactive report (findings, remediation plan, engine recommendations, coverage matrix)
html = open(out["paths"]["html"], encoding="utf-8").read()
try:
    displayHTML(html)                       # Spark notebooks
except NameError:
    from IPython.display import HTML, display
    display(HTML(html))                     # Python notebooks
print("Download or open the file from the lakehouse:", out["paths"]["html"])

In [ ]:
# 7) Optional: persist to Delta tables so you can build a Power BI report on top (needs a default lakehouse + Spark)
if SAVE_TO_TABLES:
    from datetime import datetime, timezone
    run_ts = datetime.now(timezone.utc).isoformat(timespec="seconds")
    for name, df in (("dfg2_scorecard", score_df), ("dfg2_findings", findings_df)):
        sdf = spark.createDataFrame(df.assign(run_utc=run_ts).astype(str))
        sdf.write.mode("append").option("mergeSchema", "true").saveAsTable(name)
        print("appended to table", name, len(df), "rows")

## Troubleshooting

| Symptom | Cause / fix |
|---|---|
| `CapacityNotActive` in DFG-000 | The workspace capacity is paused. Resume it; refresh history can still be read but definitions cannot. |
| HTTP 401/403 | You are not Contributor on that workspace, or the tenant blocks the Fabric REST API for your identity. |
| `pip install` fails | Outbound internet is blocked. Upload the `Analyzer` folder to the lakehouse and use the `sys.path` alternative in cell 1. |
| Non-CI/CD dataflows have no score | Set `INCLUDE_LEGACY = True`, or upgrade them to Dataflow Gen2 (CI/CD). |
| Want it on a schedule | Add this notebook to a Data Pipeline (Notebook activity) and set `SAVE_TO_TABLES = True` to track the score over time. |